# WAXAL ASR - Whisper Large-V3 Fine-Tuning with LoRA

Fine-tunes **Whisper Large-V3** on WaxalNLP training data using **LoRA** (parameter-efficient)
for Luganda, Lingala, and Shona speech recognition.

**Requirements:**
- Training parquet files in `data/` (~6.9 GB total)
- GPU with >= 8GB VRAM (RTX 4060 Laptop works with gradient checkpointing)

**Training data download links:**
```
# Luganda (5 shards, ~1.9 GB)
https://huggingface.co/datasets/google/WaxalNLP/resolve/main/data/ASR/lug/lug-train-00000.parquet
https://huggingface.co/datasets/google/WaxalNLP/resolve/main/data/ASR/lug/lug-train-00001.parquet
https://huggingface.co/datasets/google/WaxalNLP/resolve/main/data/ASR/lug/lug-train-00002.parquet
https://huggingface.co/datasets/google/WaxalNLP/resolve/main/data/ASR/lug/lug-train-00003.parquet
https://huggingface.co/datasets/google/WaxalNLP/resolve/main/data/ASR/lug/lug-train-00004.parquet

# Luganda validation (1 shard)
https://huggingface.co/datasets/google/WaxalNLP/resolve/main/data/ASR/lug/lug-validation-00000.parquet

# Lingala train + validation
https://huggingface.co/datasets/google/WaxalNLP/resolve/main/data/ASR/lin/lin-train-00000.parquet
... (check HuggingFace for all shards)
https://huggingface.co/datasets/google/WaxalNLP/resolve/main/data/ASR/lin/lin-validation-00000.parquet

# Shona train + validation
https://huggingface.co/datasets/google/WaxalNLP/resolve/main/data/ASR/sna/sna-train-00000.parquet
... (check HuggingFace for all shards)
https://huggingface.co/datasets/google/WaxalNLP/resolve/main/data/ASR/sna/sna-validation-00000.parquet
```

Place all parquet files in the `data/` folder.

## 1. Install Dependencies

In [ ]:
!pip install -q jiwer soundfile "datasets==3.2.0" transformers peft accelerate
print("Dependencies installed.")

## 2. Setup & Imports

In [ ]:
import os, csv, gc
from pathlib import Path

import datasets
import numpy as np
import torch
import transformers
import peft
import jiwer
from tqdm.auto import tqdm

PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"

# Load HF token from .env
env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    for line in env_file.read_text().strip().splitlines():
        if "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()
    print("HF_TOKEN loaded from .env")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Configuration

In [ ]:
# Model
MODEL_ID = "openai/whisper-large-v3"
SAMPLE_RATE = 16_000

# Languages
LANGUAGES = ["lug", "lin", "sna"]
WHISPER_LANG_MAP = {
    "lug": "swahili",   # Luganda not in Whisper vocab, Swahili is closest
    "lin": "lingala",
    "sna": "shona",
}

# LoRA config
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# Training
MAX_STEPS = 500             # Increase for better results (1000-3000)
BATCH_SIZE = 2              # Fits 8GB VRAM with gradient checkpointing
GRAD_ACCUM = 8              # Effective batch size = 2 * 8 = 16
LEARNING_RATE = 1e-4        # Higher LR is fine with LoRA
WARMUP_STEPS = 50
EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 25
MAX_NEW_TOKENS = 225

# Paths
OUTPUT_DIR = str(PROJECT_ROOT / "experiments" / "finetuned")
CHECKPOINT_DIR = PROJECT_ROOT / "experiments" / "finetuned" / "checkpoints" / "final"

print("Configuration set.")
print(f"  Model: {MODEL_ID}")
print(f"  LoRA: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training: {MAX_STEPS} steps, bs={BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM} effective")
print(f"  LR: {LEARNING_RATE}")

## 4. Load Training & Validation Data from Local Parquet Files

In [ ]:
def load_parquet_split(data_dir, languages, split):
    """Load all parquet shards for a given split across languages."""
    all_datasets = []
    for lang in languages:
        shards = sorted(data_dir.glob(f"{lang}-{split}-*.parquet"))
        if not shards:
            print(f"  WARNING: No {split} parquets for {lang}")
            continue
        print(f"  {lang}/{split}: {len(shards)} shard(s) ...", end=" ", flush=True)
        shard_datasets = [datasets.Dataset.from_parquet(str(s)) for s in shards]
        ds = datasets.concatenate_datasets(shard_datasets) if len(shard_datasets) > 1 else shard_datasets[0]
        ds = ds.cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE))
        all_datasets.append(ds)
        print(f"OK ({len(ds)} examples)")
    
    if not all_datasets:
        return None
    return datasets.concatenate_datasets(all_datasets) if len(all_datasets) > 1 else all_datasets[0]

print("Loading training data...")
train_ds = load_parquet_split(DATA_DIR, LANGUAGES, "train")
print(f"\nTotal training examples: {len(train_ds)}")

print("\nLoading validation data...")
val_ds = load_parquet_split(DATA_DIR, LANGUAGES, "validation")
if val_ds:
    print(f"Total validation examples: {len(val_ds)}")
else:
    print("No validation data found. Will skip evaluation during training.")

## 5. Preprocess Datasets

Extract mel spectrograms and tokenize transcriptions.

In [ ]:
print("Loading processor...")
processor = transformers.WhisperProcessor.from_pretrained(MODEL_ID)

def preprocess_batch(batch):
    """Extract mel features and tokenize labels for a batch."""
    all_features = []
    all_labels = []
    for i in range(len(batch["transcription"])):
        audio_array = np.asarray(batch["audio"][i]["array"], dtype=np.float32)
        features = processor.feature_extractor(
            audio_array, sampling_rate=SAMPLE_RATE, return_tensors="np"
        ).input_features[0]
        labels = processor.tokenizer(
            batch["transcription"][i], return_tensors="np"
        ).input_ids[0]
        all_features.append(features.tolist())
        all_labels.append(labels.tolist())
    return {"input_features": all_features, "labels": all_labels}

# Determine columns to remove
cols_to_remove = [c for c in train_ds.column_names if c not in {"input_features", "labels"}]

print("Preprocessing training data...")
train_ds = train_ds.map(
    preprocess_batch, batched=True, batch_size=16,
    remove_columns=cols_to_remove,
    desc="Preprocess train",
)
print(f"Training set ready: {len(train_ds)} examples")

if val_ds:
    val_cols = [c for c in val_ds.column_names if c not in {"input_features", "labels"}]
    print("Preprocessing validation data...")
    val_ds = val_ds.map(
        preprocess_batch, batched=True, batch_size=16,
        remove_columns=val_cols,
        desc="Preprocess val",
    )
    # Take a fixed subset for faster eval
    val_subset = val_ds.select(range(min(200, len(val_ds))))
    print(f"Validation set ready: {len(val_subset)} examples")
else:
    val_subset = None

## 6. Load Model with LoRA

In [ ]:
print(f"Loading {MODEL_ID}...")
model = transformers.WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
)

# Disable forced decoder IDs
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

# Enable gradient checkpointing to fit in 8GB VRAM
model.gradient_checkpointing_enable()
model.config.use_cache = False  # Required when gradient checkpointing is on

# Apply LoRA
lora_config = peft.LoraConfig(
    task_type=peft.TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
)

model = peft.get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available():
    print(f"GPU memory after model load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 7. Data Collator & Metrics

In [ ]:
class WhisperDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_feats = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_feats, return_tensors="pt")

        label_feats = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_feats, return_tensors="pt")
        labels = labels_batch["input_ids"]
        labels = labels.masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = WhisperDataCollator(processor)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pred_lower = [p.lower().strip() for p in pred_str]
    label_lower = [l.lower().strip() for l in label_str]
    pairs = [(r, p) for r, p in zip(label_lower, pred_lower) if r]
    if not pairs:
        return {"wer": 1.0, "cer": 1.0}
    refs, preds = zip(*pairs)
    return {
        "wer": jiwer.wer(list(refs), list(preds)),
        "cer": jiwer.cer(list(refs), list(preds)),
    }

print("Data collator and metrics ready.")

## 8. Train

In [ ]:
training_args = transformers.Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy="steps" if val_subset else "no",
    eval_steps=EVAL_STEPS if val_subset else None,
    predict_with_generate=True,
    generation_max_length=MAX_NEW_TOKENS,
    fp16=True,
    report_to="none",
    seed=42,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    label_names=["labels"],
    load_best_model_at_end=True if val_subset else False,
    metric_for_best_model="wer" if val_subset else None,
    greater_is_better=False if val_subset else None,
    save_total_limit=3,
)

trainer = transformers.Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds.shuffle(seed=42),
    eval_dataset=val_subset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

print(f"Training for {MAX_STEPS} steps...")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Training loss: {train_result.training_loss:.4f}")
if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 9. Save Fine-Tuned Model

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Save LoRA adapter weights (small, ~50-100MB)
model.save_pretrained(str(CHECKPOINT_DIR))
processor.save_pretrained(str(CHECKPOINT_DIR))

print(f"Fine-tuned model saved to: {CHECKPOINT_DIR}")
print(f"Adapter files:")
for f in sorted(CHECKPOINT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.1f} MB)")

## 10. Generate Submission with Fine-Tuned Model

Loads test audio from local parquet files and runs inference with the fine-tuned model.

In [ ]:
# Re-enable cache for faster inference
model.config.use_cache = True
model.eval()

# Load test data
print("Loading test data...")
test_data = {}
for lang in LANGUAGES:
    shards = sorted(DATA_DIR.glob(f"{lang}-test-*.parquet"))
    if not shards:
        print(f"  WARNING: No test parquets for {lang}")
        continue
    shard_datasets = [datasets.Dataset.from_parquet(str(s)) for s in shards]
    ds = datasets.concatenate_datasets(shard_datasets) if len(shard_datasets) > 1 else shard_datasets[0]
    ds = ds.cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE))
    test_data[lang] = ds
    print(f"  {lang}: {len(ds)} test examples")

# Read test IDs
test_ids = []
with open(PROJECT_ROOT / "Test.csv", "r", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        test_ids.append(row["ID"])

lang_to_ids = {}
for tid in test_ids:
    lang = tid.split("_")[0]
    lang_to_ids.setdefault(lang, []).append(tid)

print(f"\nTest set: {len(test_ids)} samples")

# Run inference
predictions = {}

for lang in LANGUAGES:
    if lang not in lang_to_ids or lang not in test_data:
        continue

    ds = test_data[lang]
    needed_ids = set(lang_to_ids[lang])
    whisper_lang = WHISPER_LANG_MAP[lang]

    # Build ID lookup
    id_lookup = {}
    for idx in range(len(ds)):
        raw_id = str(ds[idx]["id"])
        full_id = f"{lang}_{raw_id}" if not raw_id.startswith(lang) else raw_id
        if full_id in needed_ids:
            id_lookup[idx] = full_id

    print(f"\n{lang}: transcribing {len(id_lookup)} samples (lang={whisper_lang})...")

    for idx in tqdm(sorted(id_lookup.keys()), desc=f"Predict {lang}"):
        example = ds[idx]
        audio_array = np.asarray(example["audio"]["array"], dtype=np.float32)

        input_features = processor.feature_extractor(
            audio_array, sampling_rate=SAMPLE_RATE, return_tensors="pt",
        ).input_features.to(device=device, dtype=torch.float16)

        with torch.no_grad():
            pred_ids = model.generate(
                input_features,
                max_new_tokens=MAX_NEW_TOKENS,
                language=whisper_lang,
                task="transcribe",
                num_beams=5,
                no_repeat_ngram_size=3,
            )

        transcript = processor.tokenizer.decode(
            pred_ids[0], skip_special_tokens=True
        ).strip()
        predictions[id_lookup[idx]] = transcript

# Write submission
submission_dir = PROJECT_ROOT / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)
submission_path = submission_dir / "submission_finetuned.csv"

with open(submission_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ID", "Target"])
    for tid in test_ids:
        writer.writerow([tid, predictions.get(tid, "")])

print(f"\nSubmission written to: {submission_path}")
print(f"Predictions: {len(predictions)} / {len(test_ids)}")

# Validate
sample_path = PROJECT_ROOT / "SampleSubmission.csv"
if sample_path.exists():
    with open(sample_path, "r", encoding="utf-8") as f:
        expected = {row["ID"] for row in csv.DictReader(f)}
    missing = expected - set(predictions.keys())
    empty = sum(1 for tid in test_ids if not predictions.get(tid, "").strip())
    if missing:
        print(f"WARNING: Missing {len(missing)} IDs")
    elif empty:
        print(f"WARNING: {empty} IDs have empty transcriptions")
    else:
        print("Submission validation PASSED")